In [1]:
import pandas as pd
import duckdb as db

from models.TitleEmbeddingModel import TitleEmbeddingModel
from models.MFModel import MFModel
from models.RankingModel import RankingModel
from sentence_transformers import SentenceTransformer

/Users/logancostello/.pyenv/versions/spotify-playlist/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Data loading from evaluate model

groups = list(range(1, 11))

test_sets = []
train_sets = []
for i in groups:
    test_sets.append({
        'group': i,
        'playlist_metadata': pd.read_parquet(f"data/test/{i}/playlist_metadata.parquet"),
        'playlist_contents': pd.read_parquet(f"data/test/{i}/playlist_contents.parquet"),
        'holdout_contents':  pd.read_parquet(f"data/test/{i}/holdout_contents.parquet"),
    })
    train_sets.append({
        'group': i,
        'playlist_metadata': pd.read_parquet(f"data/train/{i}/playlist_metadata.parquet"),
        'playlist_contents': pd.read_parquet(f"data/train/{i}/playlist_contents.parquet"),
        'holdout_contents':  pd.read_parquet(f"data/train/{i}/holdout_contents.parquet"),
    })

ranker_train_metadata = pd.concat([ts['playlist_metadata'] for ts in train_sets], ignore_index=True)
ranker_train_contents = pd.concat([ts['playlist_contents'] for ts in train_sets], ignore_index=True)
ranker_train_holdouts = pd.concat([ts['holdout_contents']  for ts in train_sets], ignore_index=True)

original_metadata = pd.read_parquet("data/original/playlist_metadata.parquet")
original_contents = pd.read_parquet("data/original/playlist_contents.parquet")
track_metadata = pd.read_parquet("data/original/track_metadata.parquet")

train_pids = ranker_train_metadata["pid"].unique()
test_pids = pd.concat([ts['playlist_metadata']['pid'] for ts in test_sets]).unique()
held_pids  = set(train_pids) | set(test_pids)

candidate_train_metadata = pd.concat([
    original_metadata[~original_metadata["pid"].isin(held_pids)],
    ranker_train_metadata,
    *[ts['playlist_metadata'] for ts in test_sets],
], ignore_index=True)

candidate_train_contents = pd.concat([
    original_contents[~original_contents["pid"].isin(held_pids)],
    ranker_train_contents,
    *[ts['playlist_contents'] for ts in test_sets], 
], ignore_index=True)


del original_contents, original_metadata


In [3]:
# models
title_embedding_model = TitleEmbeddingModel()
mf_model = MFModel()
ranking_model = RankingModel(mf_model, title_embedding_model)

models = [title_embedding_model, mf_model, ranking_model]

In [4]:
# train models
for model in models:
    if model.is_ranker:
        model.train(ranker_train_metadata, ranker_train_contents, ranker_train_holdouts, track_metadata, candidate_train_metadata, candidate_train_contents)
    else:
        model.train(candidate_train_metadata, candidate_train_contents, track_metadata)

Save file found — loading TitleEmbedding model instead of retraining.
TitleEmbedding model loaded from saved_models/title_embedding_model/
Save file found — loading MF model instead of retraining.
MF model loaded from saved_models/mf_model/
Generating candidates...
Generating co-occurrence candidates...
Building co-occurrence index for 80,944 new seeds (0 already cached)...
  0/80,944 seeds processed...
  1,000/80,944 seeds processed...
  2,000/80,944 seeds processed...
  3,000/80,944 seeds processed...
  4,000/80,944 seeds processed...
  5,000/80,944 seeds processed...
  6,000/80,944 seeds processed...
  7,000/80,944 seeds processed...
  8,000/80,944 seeds processed...
  9,000/80,944 seeds processed...
  10,000/80,944 seeds processed...
  11,000/80,944 seeds processed...
  12,000/80,944 seeds processed...
  13,000/80,944 seeds processed...
  14,000/80,944 seeds processed...
  15,000/80,944 seeds processed...
  16,000/80,944 seeds processed...
  17,000/80,944 seeds processed...
  18,00

In [10]:
def build_custom_playlist(title, tracks, track_metadata, encoder, is_random_order=False, pid=-1):
    tm = track_metadata.copy()
    tm["track_name_lower"]  = tm["track_name"].str.lower()
    tm["artist_name_lower"] = tm["artist_name"].str.lower()

    matched_uris = []
    unmatched    = []

    for track_name, artist_name in tracks:
        mask = (
            tm["track_name_lower"].str.contains(track_name.lower(), regex=False) &
            tm["artist_name_lower"].str.contains(artist_name.lower(), regex=False)
        )
        hits = tm[mask]
        if hits.empty:
            unmatched.append((track_name, artist_name))
        else:
            matched_uris.append(hits.iloc[0]["track_uri"])

    if unmatched:
        print(f"No match found for {len(unmatched)} track(s):")
        for t, a in unmatched:
            print(f"  - '{t}' by '{a}'")

    print(title)
    for uri in matched_uris:
        row = tm[tm["track_uri"] == uri].iloc[0]
        print(f"  {row['track_name']} — {row['artist_name']}")

    embedding = encoder.encode(title)

    contents = pd.DataFrame({
        "pid":       pid,
        "track_uri": matched_uris,
        "position":  range(len(matched_uris)),
    })

    metadata = pd.DataFrame([{
        "pid":                   pid,
        "name":                  title,
        "num_tracks":            len(matched_uris),
        "title_bert_embeddings": embedding,
        "num_samples":           len(matched_uris),
        "has_title":             title!="",
        "random_order":          is_random_order,
        "num_holdouts":          0,
    }])

    return metadata, contents

In [85]:
encoder = SentenceTransformer("all-MiniLM-L6-v2")

my_tracks = [
    ("The Killchain", "Bolt Thrower")
]

my_metadata, my_contents = build_custom_playlist(
    title          = "",
    tracks         = my_tracks,
    track_metadata = track_metadata,
    encoder        = encoder,
    is_random_order= False,
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8305.87it/s]



  The Killchain — Bolt Thrower


In [86]:
# title recs
recs = title_embedding_model.predict(my_metadata, my_contents, track_metadata, n_recs=25, g_num=-1)
recs.merge(track_metadata[["track_uri","track_name","artist_name"]], on="track_uri")[["track_name", "artist_name"]]


,track_name,artist_name
0,Broccoli (feat. Lil Yachty),DRAM
1,Needed Me,Rihanna
2,One Dance,Drake
3,No Role Modelz,J. Cole
4,The Hills,The Weeknd
5,Hotline Bling,Drake
6,Don't,Bryson Tiller
7,Black Beatles,Rae Sremmurd
8,Caroline,Aminé
9,Trap Queen,Fetty Wap


In [87]:
# mf recs
recs = mf_model.predict(my_metadata, my_contents, track_metadata, n_recs=25, g_num=-1)
recs.merge(track_metadata[["track_uri","track_name","artist_name"]], on="track_uri")[["track_name", "artist_name"]]


,track_name,artist_name
0,Midnight City,M83
1,Float On,Modest Mouse
2,Electric Love,BØRNS
3,Shape of You,Ed Sheeran
4,Skinny Love,Bon Iver
5,Say You Won't Let Go,James Arthur
6,Intro,The xx
7,The Less I Know The Better,Tame Impala
8,Chocolate,The 1975
9,Raining Blood,Slayer


In [88]:
# ranker recs
recs = ranking_model.predict(my_metadata, my_contents, track_metadata, 25,-1,candidate_train_metadata,candidate_train_contents)
recs.merge(track_metadata[["track_uri","track_name","artist_name"]], on="track_uri")[["track_name", "artist_name"]]


Generating candidates...
Generating co-occurrence candidates...
Building features...


,track_name,artist_name
0,Anti-Tank (Dead Armour),Bolt Thrower
1,When Cannons Fade,Bolt Thrower
2,At First Light,Bolt Thrower
3,Salvo,Bolt Thrower
4,Lament,Bolt Thrower
5,Destructive Infinity,Bolt Thrower
6,Sixth Chapter,Bolt Thrower
7,7th Offensive,Bolt Thrower
8,Rebirth Of Humanity,Bolt Thrower
9,For Victory,Bolt Thrower
